- [ ] import combined_corpus.xlsx
- [ ] fill "" with NaN
- [ ] title search with openalex using this api:https://api.openalex.org/works?filter=title.search:
- [ ] get DOI, is_oa, license, version, pdf_url, openalex_id, URL

it is possible to limit response with select:<br>
&select=doi,best_oa_location,versions

In [10]:
import pandas as pd
import time
import requests
from urllib.parse import quote

Getting the basic function to work

In [24]:
def get_openalex_data(title):
    base_URL = "https://api.openalex.org/works"
    
    # Properly encode the title with quotes
    encoded_title = quote(f'"{title}"') # the encoded title with the " is due to the way that zotero exports titles
    punctuation_to_replace = ":;?-'"
    for char in punctuation_to_replace:
        encoded_title = encoded_title.replace(char, " ")
    full_url = f"{base_URL}?mailto=pnriddle@dal.ca&filter=title.search:{encoded_title}"

    try:
        response = requests.get(full_url)
        response.raise_for_status() #raise exception for bad status codes
        data = response.json()

        # check if results
        if data['results']:
            result = data['results'][0] # take first result - can compare later or set score limit

            return {
                'doi': result.get('doi'),
                'title':result.get('title'),
                'relevance_score':result.get('relevance_score'),
                'is_oa': result.get('best_oa_location').get('is_oa'),
                'license': result.get('best_oa_location').get('license'),
                'openalex_id': result.get('id'),
                'url': result.get('best_oa_location').get('pdf_url')
            }
        return None

    except requests.exceptions.RequestException as e:
        print(f"Ah nah, buddy, error fetching data for title {title}: {e}")
        return None

# Test the function
test_title = "A balancing act: how risk-mitigation strategies employed by 'users' explain the privacy paradox on social media?"
result = get_openalex_data(test_title)

print(f"For title searched: {test_title}")
for k,v in result.items():
    print(f"{k}:{v}")

For title searched: A balancing act: how risk-mitigation strategies employed by 'users' explain the privacy paradox on social media?
doi:https://doi.org/10.1080/0144929x.2022.2152366
title:A balancing act: how risk mitigation strategies employed by users explain the privacy paradox on social media
relevance_score:263.92255
is_oa:True
license:cc-by-nc-nd
openalex_id:https://openalex.org/W4311649162
url:https://www.tandfonline.com/doi/pdf/10.1080/0144929X.2022.2152366?needAccess=true


# This cell works!
Be warned - it takes a little while to run all 500+ titles. <br>
The intent of this is to:
- find DOIs and or openalex_ids not previously identified. 
- start gathering the data on open access status and licenses

In [27]:
# v2

def get_openalex_data(title):
    base_URL = "https://api.openalex.org/works"
    
    # Properly encode the title with quotes
    encoded_title = quote(f'{title}') # the quotes in quotes is needed and intentional
    punctuation_to_replace = ":;?-.,'"
    for char in punctuation_to_replace:
        encoded_title = encoded_title.replace(char, " ")
    full_url = f"{base_URL}?mailto=pnriddle@dal.ca&filter=title.search:{encoded_title}"

    try:
        response = requests.get(full_url)
        response.raise_for_status() # error handling
        data = response.json()

        if data['results']:
            result = data['results'][0] # takes just the first match 

            #safely get best_oa_location data
            best_oa = result.get('best_oa_location') or {}

            return {
                'doi': result.get('doi'),
                'orig_title':title,
                'found_title':result.get('title'),
                'relevance_score':result.get('relevance_score'),
                'is_oa': best_oa.get('is_oa'),
                'license': best_oa.get('license'),
                'openalex_id': result.get('id'),
                'url': best_oa.get('pdf_url')
            }
        return None

    except requests.exceptions.RequestException as e:
        print(f"Ah nah, buddy, error fetching data for title {title}: {e}")
        return None
    except Exception as e:
        print(f"oh buddy, unexpected error for {title}: {e}")
        return None

# Read titles from Excel file
# can specify which sheet to read
file_path = "/Users/poppyriddle/Documents/LNS/corpus/combined_outputs_2.csv"

try:
    input_df = pd.read_csv(file_path, delimiter="|", encoding="UTF-8")  # note the delimited
except Exception as e:
    print(f"oh, crickets! something went wrong: {e}")

# Create a list to store results
results = []

# Process each title
for index, row in input_df.iterrows():
    try:
        title = str(row['Title']) #just in case non-string values
        print(f"Processing title {index + 1} of {len(input_df)}: {title}")
    
        #skip empties
        if pd.isna(title) or title.strip()=="":
            print(f"skipped this one: {index+1}")
            continue

        # Get data for this title
        result = get_openalex_data(title)
    
        if result:
            results.append(result)
        else:
            print(f"No results for: {title}")
    
        # Add a small delay to avoid hitting API rate limits
        time.sleep(0.5)
    
    except Exception as e:
        print(f"oh betty, error at {index+1}: {e}")
        continue

try:
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)

    # Display first few rows of results
    print("\nFirst few rows of results:")
    print(results_df.head())

    # Save results to Excel
    results_df.to_csv('LNS_search_results_REV_2.csv', index=False)
    print("\nResults saved to 'LNS_search_results_REV_2.csv'")

    # Print some basic statistics
    print(f"\nTotal titles processed: {len(input_df)}")
    print(f"Successful matches found: {len(results_df)}")

except Exception as e:
    print(f"aw snap, something went wrong saving :{e}")

Processing title 1 of 503: Adults With Inadequate Literacy Skills
No results for: Adults With Inadequate Literacy Skills
Processing title 2 of 503: High levels of household food insecurity on the Navajo Nation
Processing title 3 of 503: Piruqsiaksaqtaaqtuq Annual Report 2021-2022
No results for: Piruqsiaksaqtaaqtuq Annual Report 2021-2022
Processing title 4 of 503: Piruqsiaksaqtaaqtuq Annual Report 2021-2022
No results for: Piruqsiaksaqtaaqtuq Annual Report 2021-2022
Processing title 5 of 503: The Daily, Wednesday, November 9, 2005. International Adult Literacy and Skills Survey
Ah nah, buddy, error fetching data for title The Daily, Wednesday, November 9, 2005. International Adult Literacy and Skills Survey: 403 Client Error: FORBIDDEN for url: https://api.openalex.org/works?mailto=pnriddle@dal.ca&filter=title.search:The%20Daily%2C%20Wednesday%2C%20November%209%2C%202005%20%20International%20Adult%20Literacy%20and%20Skills%20Survey
No results for: The Daily, Wednesday, November 9, 200

In [28]:
try:
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)

    # Display first few rows of results
    print("\nFirst few rows of results:")
    print(results_df.head())


    # Print some basic statistics
    print(f"\nTotal titles processed: {len(input_df)}")
    print(f"Successful matches found: {len(results_df)}")

except Exception as e:
    print(f"aw snap, something went wrong saving :{e}")


First few rows of results:
                                          doi  ...                                                url
0   https://doi.org/10.1017/s1368980012005630  ...  https://www.cambridge.org/core/services/aop-ca...
1            https://doi.org/10.1037/a0029356  ...                                               None
2  https://doi.org/10.1177/002188637601200414  ...                                               None
3  https://doi.org/10.1177/002188637601200414  ...                                               None
4      https://doi.org/10.5281/zenodo.7375224  ...  https://zenodo.org/records/7375224/files/B-115...

[5 rows x 8 columns]

Total titles processed: 503
Successful matches found: 277


# Next:
- [x] merge on DOI
- [x] check if any are missing from input_df

In [29]:
print(input_df.columns)
print(results_df.columns)

Index(['﻿"Key"', 'Item Type', 'Publication Year', 'Author', 'Title',
       'Publication Title', 'ISBN', 'ISSN', 'DOI', 'Url', 'Abstract Note',
       'Date', 'Date Added', 'Date Modified', 'Access Date', 'Pages',
       'Num Pages', 'Issue', 'Volume', 'Number Of Volumes',
       'Journal Abbreviation', 'Short Title', 'Series', 'Series Number',
       'Series Text', 'Series Title', 'Publisher', 'Place', 'Language',
       'Rights', 'Type', 'Archive', 'Archive Location', 'Library Catalog',
       'Call Number', 'Extra', 'Notes', 'File Attachments', 'Link Attachments',
       'Manual Tags', 'Automatic Tags', 'Editor', 'Series Editor',
       'Translator', 'Contributor', 'Attorney Agent', 'Book Author',
       'Cast Member', 'Commenter', 'Composer', 'Cosponsor', 'Counsel',
       'Interviewer', 'Producer', 'Recipient', 'Reviewed Author',
       'Scriptwriter', 'Words By', 'Guest', 'Number', 'Edition',
       'Running Time', 'Scale', 'Medium', 'Artwork Size', 'Filing Date',
       'Applica

In [33]:
# check if DOIs in input_df are in results_df.
# if not, add dois,title from input_df to results_df

#make a copy of results_df
results_df_2 = results_df.copy(deep=True)

#check for missing DOIs in results_df
missing_dois = results_df_2['doi'].isna()

#count missing
missing_count = missing_dois.sum()
print(f"number of missing DOIs in results_df_2: {missing_count}")

# if missing DOIs, update using inputs_df
if missing_dois.any():
    # merge the dataframes to get the missing information
    # left column is from results_df_2, right col is from input_df
    results_df_2 = results_df_2.merge(
        input_df[['DOI', 'Title']], 
        how='left',
        left_on='doi',  
        right_on='DOI'
    )
    
    #count how many found in input_df
    #found_in_input = results_df_2.loc[missing_dois, 'DOI'].notna().sum()
    #print(f"number missing found in input_df: {found_in_input}")

    # update only the missing DOIs and titles
    results_df_2.loc[missing_dois, 'doi'] = results_df_2.loc[missing_dois, 'DOI']
    results_df_2.loc[missing_dois, 'orig_title'] = results_df_2.loc[missing_dois, 'Title']
    
    # clean up unnecessary cols
    results_df_2 = results_df_2.drop(['DOI', 'Title'], axis=1)

    # count how many still missing or don't have DOIs
    still_missing = results_df_2['doi'].isna().sum()
    print(f"number still missing DOIs: {still_missing}")

    results_df_2

number of missing DOIs in results_df_2: 34


AssertionError: 

# check to see if titles match ok 
- [ ] Flag those that don't with Levenshtein distance ratio
- [ ] manually check flagged ones for manual removal



In [36]:
from Levenshtein import ratio

In [37]:
# func to compare title and return similarity ratio
def compare_titles(title1,title2):
    #error handling for NaN
    if pd.isna(title1) or pd.isna(title2):
        return 0
    
    # convert to strings
    title1 = str(title1).lower().strip()
    title2 = str(title2).lower().strip()

    # similarity ratio
    return ratio(title1,title2)

#apply to results_df_2 
results_df['title_similarity'] = results_df.apply(lambda x: compare_titles(x['orig_title'],x['found_title']), axis=1)

#create flag: 1=match, 0=no match based on some ratio threshold
results_df['title_check_flag'] = (results_df['title_similarity']>= 0.85).astype(int)

print(f"number of matching titles for a given threshold: {(results_df['title_check_flag']==1).sum()}")
print(f"\nNumber of non-matching titles for a given threshold: {(results_df['title_check_flag']==0).sum()}")

number of matching titles for a given threshold: 247

Number of non-matching titles for a given threshold: 30


# cleanup
- [x] remove resolver from 'doi'

In [38]:
#remove resolver from 'doi' column in results_df
results_df['doi'] = results_df['doi'].str.lstrip('https://doi.org/')


# summarize
- [ ]how many title matches did I get?
- [ ]how many are is_oa=True?
- [ ] how many of each license type?

In [39]:
#summarize
print("********** gimme some stats! **********")

# count matching titles
#matching_titles = results_df['title_check_flag'].value_counts()
#print(f"matching titles (flag=1): {matching_titles.get(1,0)}")
#print(f"non-matching titles (flag=0): {matching_titles.get(0,0)}")
#print(f"percent matching: {(matching_titles.get(1,0)/len(results_df)*100):.1f}%")

# how many have DOIs
have_doi = results_df['doi'].notna().sum()
print(f"\nthose with DOI: {have_doi}")
percentage = (have_doi/len(results_df)*100)
print(f"percentage with DOI: {percentage:.1f}%")

#count is_oa
oa_count = results_df['is_oa'].value_counts()
print(f"\nopen access true: {oa_count.get(True,0)}")
print(f"open access false: {oa_count.get(False,0)}")
print(f"percentage OA: {(oa_count.get(True,0)/len(results_df)*100):.1f}%")

# license
null_license = results_df['license'].isna().sum()
if null_license >0:
    percentage = (null_license/len(results_df)*100)
    print(f"missing licenses: {null_license} or {percentage:.1f}%")

# get license types and sums
license_types = results_df['license'].value_counts()
print("\n License type counts")
for k,v in license_types.items():
    percentage = (v/len(results_df)*100)
    print(f"{k}: {v} or {percentage:.1f}%")



********** gimme some stats! **********

those with DOI: 243
percentage with DOI: 87.7%

open access true: 113
open access false: 0
percentage OA: 40.8%
missing licenses: 197 or 71.1%

 License type counts
cc-by: 40 or 14.4%
cc-by-nc-nd: 17 or 6.1%
cc-by-nc-sa: 7 or 2.5%
other-oa: 7 or 2.5%
cc-by-nc: 4 or 1.4%
cc-by-sa: 3 or 1.1%
publisher-specific-oa: 2 or 0.7%


# match up results_df_2 to LNS_corpus
I save it in csv format so that its human readable. I need to open this in something else so that I can easily compare the two and edit the file for titles that do not match. Otherwise we get false positives.


In [41]:
file_location_to_save = ""
results_df.to_csv("LNS_final_corpus_rev2.csv", encoding="UTF-8")


# rerun titles to get the OpenAlex ID 
Open alex found an ID for almost everyone. 
- [ ] get metadata from OpenAlex for each work id
- [ ] export again - but now export in a way that preserves the data structure using pickle

In [44]:
# for openalex_id in results_df get metadata record from openalex api

# for openalex_id in results_df get metadata record from openalex api
def get_openalex_metadata(id):
    URL = f"https://api.openalex.org/works?mailto=pnriddle@dal.ca&filter=ids.openalex:{id}"

    try:
        response = requests.get(URL)
        response.raise_for_status()  # error handling
        data = response.json()

        if data['results']:
            result = data['results'][0]
        return result
    #this exception is specifically from the requests library
    except requests.exceptions.RequestException as e:
        print(f"Ah nah, buddy, error fetching data for title {id}: {e}")
        return None
        # for all others
    except Exception as e:
        print(f"oh buddy, unexpected error for {id}: {e}")
        return None

In [167]:
# strip https://openalex.org/ from each id

# get the manually reviewed csv file from above
file_name = "/Users/poppyriddle/Documents/LNS/corpus/LNS_final_corpus_rev2.csv"
reviewed_df = pd.read_csv(file_name,encoding="UTF-8")

# empty list
metadata_list = []
# Iterate through each row in the DataFrame
for index, row in reviewed_df.iterrows():
    openalex_id = row['openalex_id'].lstrip("https://openalex.org/")
    metadata = get_openalex_metadata(openalex_id)

    if metadata:
        # append to a list
        metadata_list.append(metadata)
        print(f" {index+1}", end=",")
    else:
        metadata_list.append(None)
        print("ah, gotta problem with that one, buddy.")
    
#extract keys
keys = list(metadata.keys())

openalex_results_df = pd.DataFrame(metadata_list, columns=keys)
print('\ndone')


 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 22

In [435]:
file_name = "LNS_openalex_full_metadata_REV2_uncleaned.pkl"
openalex_results_df.to_pickle(file_name)
print(Fore.LIGHTCYAN_EX + f"saved to pickle format: {file_name}")

saved to pickle format: LNS_openalex_full_metadata_REV2_uncleaned.pkl


## Data Cleaning
expanding nested lists and dictionaries into rows which will be easier to explore
- [x] 'primary_location'
- [x] 'open_access'
- [x] reconstruct 'abstract_inverted_index'
- [ ] 'best_oa_location'

In [436]:
print(openalex_results_df['primary_location'].iloc[0].keys())

#extract primary location column in a 'data-to-columns' approach
primary_loc_dict = openalex_results_df['primary_location'].to_dict()
primary_loc_df = pd.DataFrame(primary_loc_dict)

#transpose
primary_loc_df = primary_loc_df.T

#merge
openalex_results_df2 = pd.concat([openalex_results_df, primary_loc_df], axis=1)


dict_keys(['is_oa', 'landing_page_url', 'pdf_url', 'source', 'license', 'license_id', 'version', 'is_accepted', 'is_published'])


In [437]:
# reconstruct abstract
#import importlib
import reconstruct_abstract
#importlib.reload(reconstruct_abstract) # only needed because I made changes to the module

openalex_results_df2['abstract'] = openalex_results_df2['abstract_inverted_index'].apply(reconstruct_abstract.main)

In [438]:
#cited_by_percentile_year
cited_by_pct_year_dict = openalex_results_df2['cited_by_percentile_year'].to_dict()
cited_by_pct_year_df = pd.DataFrame(cited_by_pct_year_dict).T

#this adds a prefix to the new column names
prefix = 'cited_by_pct_year'
cited_by_pct_year_df.columns = [prefix + str(col) for col in cited_by_pct_year_df.columns]

openalex_results_df2 = pd.concat([openalex_results_df2,cited_by_pct_year_df], axis=1)

In [439]:
# ids
ids_dict = openalex_results_df2['ids'].to_dict()
ids_df = pd.DataFrame(ids_dict).T

prefix = 'ids_'
ids_df.columns = [prefix + str(col) for col in ids_df.columns]

openalex_results_df2 = pd.concat([openalex_results_df2,ids_df], axis=1)

In [440]:

df_authorships = openalex_results_df[['authorships','id']].copy()
df_authorships.head(2)


for index, row in df_authorships.iterrows():
    authors = []
    for author_info in row['authorships']:
        author_name = author_info.get('author', {}).get('display_name', '')
        if author_name:
            authors.append(author_name)
    df_authorships.at[index, 'authorships'] = authors
    df_authorships.at[index, 'new_column'] = row['id']  # Assuming 'id' is the new column

print(df_authorships)
openalex_results_df2 = pd.merge(openalex_results_df2,df_authorships, how='left', on='id')


                                           authorships  ...                        new_column
0    [Marla Pardilla, Divya Prasad, Sonali Suratkar...  ...  https://openalex.org/W2170834431
1             [Anatoliy Gruzd, Ángel Hernández-García]  ...  https://openalex.org/W4311649162
2    [Philips Ayeni, Blessed O. Agbaje, Maria Tippler]  ...  https://openalex.org/W3199214563
3    [Marc Stoeckle, James E. Murphy, Bartlomiej A....  ...  https://openalex.org/W4283837859
4    [Nena Schvaneveldt, Sean M. Stone, Erica R. Br...  ...  https://openalex.org/W3162879256
..                                                 ...  ...                               ...
248                [Christopher Prechotko, Dale Kirby]  ...  https://openalex.org/W4379055648
249                 [Suzanne Smythe, Sherry Breshears]  ...  https://openalex.org/W2600607963
250                                       [Carole Roy]  ...  https://openalex.org/W1500331699
251                [Casey Burkholder, Marianne Filion]  ... 

In [441]:
print(len(openalex_results_df['authorships'].iloc[0]))
#print(openalex_results_df['authorships'].iloc[0])
print(openalex_results_df['authorships'].iloc[0][0].keys())
print(openalex_results_df['authorships'].iloc[0][0].get('institutions'))
print(openalex_results_df['authorships'].iloc[0][0].get('institutions')[0].get('display_name'))
print(openalex_results_df['authorships'].iloc[0][0].get('institutions')[0].get('ror'))
print(len(df_institutions))
df_institutions = openalex_results_df[['authorships', 'id']].copy()
big_list = []
for index, row in df_institutions.iterrows():
    institutions = []
    for each in row['authorships']:
        try:
            first_level = each.get('institutions', '')
            for inst in first_level:
                inst_name = inst.get('display_name', '')
                ror_id = inst.get('ror', '')
            if inst_name:
                institutions.append([inst_name,ror_id])        
        except KeyError as e:
            institutions.append([])
    big_list.append(institutions)
    
print(f"big_list length: {len(big_list)}")
df_institutions['institution_names'] = big_list
df_institutions.drop(columns=['authorships'], inplace=True)

openalex_results_df2 = pd.merge(openalex_results_df2,df_institutions, on='id', how='left')


4
dict_keys(['author_position', 'author', 'institutions', 'countries', 'is_corresponding', 'raw_author_name', 'raw_affiliation_strings', 'affiliations'])
[{'id': 'https://openalex.org/I145311948', 'display_name': 'Johns Hopkins University', 'ror': 'https://ror.org/00za53h95', 'country_code': 'US', 'type': 'funder', 'lineage': ['https://openalex.org/I145311948']}]
Johns Hopkins University
https://ror.org/00za53h95
253
big_list length: 253


In [444]:
#openalex_results_df.drop(openalex_results_df.columns[-8:], axis=1, inplace=True)
print(openalex_results_df2.columns)

Index(['id', 'doi', 'title', 'display_name', 'publication_year',
       'publication_date', 'ids', 'language', 'primary_location', 'type',
       'type_crossref', 'indexed_in', 'open_access', 'authorships_x',
       'institution_assertions', 'countries_distinct_count',
       'institutions_distinct_count', 'corresponding_author_ids',
       'corresponding_institution_ids', 'apc_list', 'apc_paid', 'fwci',
       'has_fulltext', 'cited_by_count', 'citation_normalized_percentile',
       'cited_by_percentile_year', 'biblio', 'is_retracted', 'is_paratext',
       'primary_topic', 'topics', 'keywords', 'concepts', 'mesh',
       'locations_count', 'locations', 'best_oa_location',
       'sustainable_development_goals', 'grants', 'datasets', 'versions',
       'referenced_works_count', 'referenced_works', 'related_works',
       'abstract_inverted_index', 'abstract_inverted_index_v3',
       'cited_by_api_url', 'counts_by_year', 'updated_date', 'created_date',
       'is_oa', 'landing_page_u

In [443]:
#openalex_results_df2.drop('new_column', axis=1, inplace=True)

In [446]:
openalex_results_df2.to_pickle("LNS_openalex_full_metadata_REV2_cleaned.pkl")

# are all the titles from Maddie's list in here?
1. Maddie sent me a list of titles for the core of the survey
2. match these to see if they are in the larger corpus from her Zotero library. 

In [458]:
# match 'title' in openalex_results_df2 to LNS_corpus.xlsx in ARCHIVE
# LNS_corpus.xlsx is a list I received from Maddie
title_df = openalex_results_df2[['title','id']]
title_df

title_maddie = pd.read_excel("/Users/poppyriddle/Documents/LNS/corpus/ARCHIVE/LNS_corpus.xlsx", sheet_name="Sheet1")
title_maddie

title_maddie['match'] = title_df['title'].isin(title_maddie['Title'])
title_maddie



,Title,match
0,Enhancing Displaced Workers' Literacy and Esse...,False
1,Enhancing Displaced Workers' Literacy and Esse...,False
2,"Young Canadians in a Wireless World, Phase IV:...",False
3,Scan of family literacy and health: Final repo...,False
4,"Read, Speak, Sing: Promoting early literacy in...",False
...,...,...
181,Cultural alloys and heterogeneous mixes: Conte...,False
182,Navigating the digital world: development of a...,False
183,Measuring skill-based health literacy in chron...,False
184,"COVID-19 vaccine uptake, conspiracy theories, ...",False
